# StyleGAN2-ADA on Kaggle — smoke run

Trains one class for 20 kimg to prove the pipeline works and measure speed.

**Accelerator** GPU T4 x2 · **Internet** On · attach the `stylegan.zip` dataset.

Steps 1–6 take ~30 min. Step 7 is an optional throughput test and needs only
steps 1, 3, 4 before it.

Steps 1–6 run on **one** GPU: `train.py` goes in-process there, while multi-GPU
hides tracebacks inside `ProcessRaisedException`.

The one thing needing a human eye: the grid in step 3 **must look like dogs**.

## 1. Setup

In [ ]:
import os, sys, json, time, pathlib, subprocess, shutil
import torch

REPO = "/kaggle/working/stylegan2-ada-pytorch"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "-q",
                "https://github.com/NVlabs/stylegan2-ada-pytorch.git", REPO], check=True)
sys.path.insert(0, REPO)
NL = chr(10)

def patch(rel, old, new):
    f = pathlib.Path(REPO) / rel
    s = f.read_text()
    assert old in s, f"patch target not found in {rel}"
    f.write_text(s.replace(old, new))

# Seven patches for torch 2.x. Reasoning in docs/stylegan.md.

# Kernels report "Failed!" after building fine.
patch("torch_utils/custom_ops.py",
      "torch.utils.cpp_extension.load(name=module_name",
      "module = torch.utils.cpp_extension.load(name=module_name")
patch("torch_utils/custom_ops.py",
      "        module = importlib.import_module(module_name)" + NL, "")

# TypeError: object.__init__() takes exactly one argument
patch("torch_utils/misc.py", "super().__init__(dataset)", "super().__init__()")

# R1 needs grid_sample's second derivative, which torch still lacks.
GSG = "torch_utils/ops/grid_sample_gradfix.py"
patch(GSG, "any(torch.__version__.startswith(x) for x in ['1.7.', '1.8.', '1.9'])",
      "True")
patch(GSG,
      "op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')" + NL +
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)",
      "op = torch.ops.aten.grid_sampler_2d_backward" + NL +
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False, [True, True])")

# batch_gpu is pinned to mb // 8 (NVlabs' rig, not ours). Default unchanged.
patch("train.py", "args.batch_gpu = spec.mb // spec.ref_gpus",
      "args.batch_gpu = int(os.environ.get('BATCH_GPU', spec.mb // spec.ref_gpus))")

# Multi-GPU only: ranks disagree on noise_const. Sync once from rank 0.
patch("training/training_loop.py",
      "    # Print network summary tables.",
      NL.join(["    if num_gpus > 1:",
               "        torch.cuda.set_device(device)",
               "        for _m in [G, D, G_ema]:",
               "            for _, _t in misc.named_params_and_buffers(_m):",
               "                torch.distributed.broadcast(_t, src=0)",
               "",
               "    # Print network summary tables."]))

NUM_GPUS = 1   # single-GPU runs in-process; DDP hides tracebacks
print("torch:", torch.__version__, "|", torch.cuda.get_device_name(0),
      "| visible gpus:", torch.cuda.device_count(), "-> using", NUM_GPUS)

## 2. CUDA kernels and the R1 path

First use of `bias_act` / `upfirdn2d` triggers an `nvcc` build — 2–5 min, silent.
Without it, training falls back to slow reference code.

The second check runs the double-backward the R1 penalty needs. It costs a
fraction of a second and fails here rather than ten minutes into training.

In [ ]:
from torch_utils.ops import bias_act, upfirdn2d, grid_sample_gradfix

t0 = time.time()
bias_act.bias_act(torch.randn(4, 8, device="cuda"),
                  torch.randn(8, device="cuda"), impl="cuda")
upfirdn2d.upfirdn2d(torch.randn(2, 3, 16, 16, device="cuda"),
                    torch.ones(4, 4, device="cuda") / 16, impl="cuda")
torch.cuda.synchronize()

ok = bias_act._init() and upfirdn2d._init()
print(f"built in {time.time()-t0:.0f}s")
print("CUDA kernels:", "compiled" if ok else "FAILED - training will be slow")

# The R1 penalty differentiates through the augment pipeline twice. This is
# that operation, in miniature.
grid_sample_gradfix.enabled = True
img = torch.randn(2, 3, 16, 16, device="cuda", requires_grad=True)
grid = torch.rand(2, 16, 16, 2, device="cuda") * 2 - 1
d = torch.nn.Conv2d(3, 1, 3, padding=1).cuda()
g = torch.autograd.grad(d(grid_sample_gradfix.grid_sample(img, grid)).sum(),
                        img, create_graph=True)[0]
g.square().sum().backward()
print("R1 double backward: OK")

## 3. Pretrained model

LSUN Dog at 256px — the closest public checkpoint to full-body creatures.

**The grid must look like dogs.** If it is noise, `--resume` did not load and
training would silently start from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import legacy, dnnlib

URL = ("https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/"
       "transfer-learning-source-nets/lsundog-res256-paper256-kimg100000-noaug.pkl")
PKL = "/kaggle/working/lsundog-res256.pkl"
if not os.path.exists(PKL):
    subprocess.run(["wget", "-q", "-O", PKL, URL], check=True)

with dnnlib.util.open_url(PKL) as f:
    G = legacy.load_network_pkl(f)["G_ema"].to("cuda")

z = torch.from_numpy(np.random.RandomState(0).randn(8, G.z_dim)).to("cuda")
with torch.no_grad():
    img = G(z, None, truncation_psi=0.7, noise_mode="const")
img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()

fig, axes = plt.subplots(1, 8, figsize=(16, 2.2))
for ax, im in zip(axes, img):
    ax.imshow(im); ax.axis("off")
plt.suptitle("must look like dogs", fontsize=12)
plt.show()
print("resolution:", G.img_resolution)

## 4. Dataset

Built by NVlabs' `dataset_tool.py`, so the zip format cannot drift.

In [ ]:
DATA = next(p.parent for p in pathlib.Path("/kaggle/input").glob("**/summary.json"))
counts = json.loads((DATA / "summary.json").read_text())["classes"]
print("classes:", {k: v["count"] for k, v in counts.items()})

CLASS = "arthropod"
ZIP = f"/kaggle/working/{CLASS}.zip"
subprocess.run([sys.executable, f"{REPO}/dataset_tool.py",
                f"--source={DATA / CLASS}", f"--dest={ZIP}"], check=True)

from training.dataset import ImageFolderDataset
ds = ImageFolderDataset(path=ZIP, use_labels=False, max_size=None, xflip=False)
print(f"{CLASS}: {len(ds)} images, {ds.image_shape}")

## 5. Train — 20 kimg

`--cfg=paper256` must match the source net or `--resume` fails on layer shapes.
~9 min. `conv2d_gradfix` warns on every convolution; those lines are filtered.

Success ends with `Exiting...`.

In [ ]:
OUT = "/kaggle/working/smoke"
cmd = [sys.executable, f"{REPO}/train.py",
       f"--outdir={OUT}", f"--data={ZIP}", f"--gpus={NUM_GPUS}",
       "--cfg=paper256", "--mirror=1", "--aug=ada", "--target=0.6",
       f"--resume={PKL}", "--snap=1", "--metrics=none", "--kimg=20"]

t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    if "conv2d_gradfix" not in line:      # expected, and thousands of them
        print(line, end="")
p.wait()

elapsed = time.time() - t0
if p.returncode != 0:
    raise SystemExit(f"training failed (exit {p.returncode})")
print(f"done in {elapsed/60:.1f} min")

## 6. Speed and results

`sec/kimg` is measured between ticks — tick 0 includes kernel compilation.

In [ ]:
run = sorted(pathlib.Path(OUT).glob("00000-*"))[-1]
ticks = [json.loads(l) for l in (run / "stats.jsonl").read_text().splitlines() if l.strip()]
get = lambda t, k: t.get(k, {}).get("mean", 0)

kimg = get(ticks[-1], "Progress/kimg")
if len(ticks) >= 3:
    sec_per_kimg = ((get(ticks[-1], "Timing/total_sec") - get(ticks[1], "Timing/total_sec"))
                    / (kimg - get(ticks[1], "Progress/kimg")))
else:
    sec_per_kimg = elapsed / kimg

print(f"sec/kimg: {sec_per_kimg:.1f}")
for b in (200, 400, 600):
    print(f"  {b} kimg per class -> {sec_per_kimg*b/3600:.1f} h")
print()
for t in ticks:
    print(f"  kimg {get(t,'Progress/kimg'):5.0f}  G {get(t,'Loss/G/loss'):7.3f}"
          f"  D {get(t,'Loss/D/loss'):7.3f}  ada_p {get(t,'Progress/augment'):.3f}")

snap = sorted(run.glob("network-snapshot-*.pkl"))[-1]
with dnnlib.util.open_url(str(snap)) as f:
    G2 = legacy.load_network_pkl(f)["G_ema"].to("cuda")
with torch.no_grad():
    img2 = G2(z, None, truncation_psi=0.7, noise_mode="const")
print(f"\nchanged from source by {(img2 - G(z, None, truncation_psi=0.7, noise_mode='const')).abs().mean():.4f}")

img2 = (img2.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()
fig, axes = plt.subplots(1, 8, figsize=(16, 2.2))
for ax, im in zip(axes, img2):
    ax.imshow(im); ax.axis("off")
plt.suptitle(f"after {kimg:.0f} kimg - still dog-like, but drifting", fontsize=12)
plt.show()

## 7. Throughput

`train.py` pins `batch_gpu = mb // ref_gpus` to NVlabs' 8-GPU rig, so steps 1–6
ran batch 64 as **8 sequential rounds of 8 images** on one T4 at 3.0 GB of 16.

`batch_size` stays 64 throughout, so gradients are identical — only chunking and
GPU count change. All three configs measured:

| config | rounds | sec/kimg | |
|---|---|---|---|
| 1 GPU, `batch_gpu=8` | 8 | 70.9 | baseline |
| 1 GPU, `batch_gpu=32` | 2 | 65.5 | 1.08x |
| **2 GPUs, `batch_gpu=32`** | 1 | **33.2** | **2.14x** |

Collapsing 8 rounds to 2 bought only 8% — the T4 was already compute-bound at
batch 8. **The second GPU was the whole win.**

In [ ]:
# Deliberately NOT filtering "_execution_engine.run_backward": that line shows
# up in real tracebacks as well as in the benign DDP warning, and a filter that
# can swallow traceback lines is exactly how the R1 failure became unreadable.
NOISE = ("conv2d_gradfix", "Grad strides", "grad.sizes()", "bucket_view.sizes()")

def timed_run(gpus, batch_gpu, outdir, kimg=12):
    cmd = [sys.executable, f"{REPO}/train.py",
           f"--outdir={outdir}", f"--data={ZIP}", f"--gpus={gpus}",
           "--cfg=paper256", "--mirror=1", "--aug=ada", "--target=0.6",
           f"--resume={PKL}", "--snap=50", "--metrics=none", f"--kimg={kimg}"]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1,
                         env=dict(os.environ, BATCH_GPU=str(batch_gpu)))
    ticks = []
    for line in p.stdout:
        if any(n in line for n in NOISE):
            continue
        if line.startswith("tick"):
            ticks.append(line)
        print(line, end="")
    p.wait()

    # sec/kimg off the LAST tick; tick 0 includes kernel compilation.
    last = [t for t in ticks if "sec/kimg" in t]
    rate = float(last[-1].split("sec/kimg")[1].split()[0]) if last else None
    print()
    print(f"--- gpus={gpus} batch_gpu={batch_gpu} rounds={64 // (batch_gpu * gpus)}"
          f" -> exit {p.returncode} ---")
    if rate:
        print(f"    sec/kimg {rate:.1f}   (baseline 70.9)   "
              f"{70.9 / rate:.2f}x   10 classes x 300 kimg = {rate * 3000 / 3600:.0f} h")
    return rate

### 7a. One GPU, `batch_gpu=32`

Two rounds instead of eight, in-process, no DDP. Measured **65.5 sec/kimg**.

In [ ]:
timed_run(gpus=1, batch_gpu=32, outdir="/kaggle/working/speed1")

### 7b. Two GPUs

Needs the `training_loop.py` buffer-sync patch from step 1: `--resume` loads on
rank 0 alone and DDP no longer syncs buffers at construction, so the ranks
disagreed on `noise_const`.

Measured **33.2 sec/kimg** — this is what Phase 3 uses.

In [ ]:
timed_run(gpus=2, batch_gpu=32, outdir="/kaggle/working/speed2")

## Done

Phase 3 runs at **33.2 sec/kimg**: 2.8 h per class at 300 kimg, and all ten
classes inside one ~30 GPU-h week.